<a href="https://colab.research.google.com/github/viviantram03/labb-1/blob/main/Lab2aml.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Setup and Preparation

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import datasets, models, transforms
import matplotlib.pyplot as plt
import time
import os
import copy

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")



Using device: cpu


## Data Augmentation and Dataloaders

In [ ]:
data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

if not os.path.exists('hymenoptera_data'):
    !wget https://download.pytorch.org/tutorial/hymenoptera_data.zip
    !unzip hymenoptera_data.zip
    !rm hymenoptera_data.zip

data_dir = 'hymenoptera_data'
image_datasets = {x: datasets.ImageFolder(os.path.join(data_dir, x), data_transforms[x]) for x in ['train', 'val']}
dataloaders = {x: torch.utils.data.DataLoader(image_datasets[x], batch_size=4, shuffle=True, num_workers=2) for x in ['train', 'val']}
dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val']}
class_names = image_datasets['train'].classes




--2026-05-05 10:16:17--  https://download.pytorch.org/tutorial/hymenoptera_data.zip
Resolving download.pytorch.org (download.pytorch.org)... 18.160.143.21, 18.160.143.101, 18.160.143.107, ...
Connecting to download.pytorch.org (download.pytorch.org)|18.160.143.21|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 47286322 (45M) [application/zip]
Saving to: ‘hymenoptera_data.zip’

hymenoptera_data.zi 100%[===================>]  45.10M   211MB/s    in 0.2s    

2026-05-05 10:16:17 (211 MB/s) - ‘hymenoptera_data.zip’ saved [47286322/47286322]

Archive:  hymenoptera_data.zip
   creating: hymenoptera_data/
   creating: hymenoptera_data/train/
   creating: hymenoptera_data/train/ants/
  inflating: hymenoptera_data/train/ants/0013035.jpg  
  inflating: hymenoptera_data/train/ants/1030023514_aad5c608f9.jpg  
  inflating: hymenoptera_data/train/ants/1095476100_3906d8afde.jpg  
  inflating: hymenoptera_data/train/ants/1099452230_d1949d3250.jpg  
  inflating: hymenoptera_da

## Design: CNN vs MLP

MLP Model

In [ ]:
class MLPModel(nn.Module):
  def __init__(self):
    super(MLPMOdel, self).__init__()
    self.flatten = nn.Flatten()
    self.fc = nn.Sequential(
        nn.Linear(224*224*3, 512),
        nn.ReLU(),
        nn.Linear(512, len(class_names))
    )

  def forward(self, x):
    x = self.flatten(x)
    return self.fc(x)

Custom CNN Model

In [ ]:
class SimpleCNN(nn.Module):
  def __init__(self):
    super(SimpleCNN, self).__init__()
    self.features = nn.Sequential (
        nn.Conv2d(3, 16, kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(2, 2),
        nn.Conv2d(16, 32, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2, 2)
    )
    self.classifier = nn.Sequential(
        nn.Linear(32*56*56, 128),
        nn.ReLU(),
        nn.Linear(128, len(class_names))
    )

  def forward(self, x):
    x = self.features(x)
    x = x.view(x.size(0), -1)
    return self.classifier(x)


## Fine-tuning Pretrained Models

Method 1: Freezing Weights (ResNet18)

In [ ]:
model_resnet = models.resnet18(pretrained=True)
for param in model_resnet.parameters():
  param.requires_grad = False

num_ftrs = model_resnet.fc.in_features
model_resnet.fc = nn.Linear(num_ftrs, len(class_names))
model_resnet = model_resnet.to(device)



/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 115MB/s]


Method 2: Reconstructing Layers (MobileNetV2)

In [ ]:
model_mobilenet = models.mobilenet_v2(pretrained=True)
num_ftrs = model_mobilenet.classifier[1].in_features
model_mobilenet.classifier[1] = nn.Linear(num_ftrs, len(class_names))
model_mobilenet = model_mobilenet.to(device)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 71.9MB/s]


## Training Loop and Results

In [44]:
def train_model(model, criterion, optimizer, scheduler=None, num_epochs=3):
  for epoch in range(num_epochs):
    print(f'Epoch {epoch}/{num_epochs - 1}')
    for phase in ['train', 'val']:
      if phase == 'train':
        model.train()
      else:
        model.eval()

      running_loss, running_corrects = 0.0, 0

      for inputs, labels in dataloaders[phase]:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        with torch.set_grad_enabled(phase == 'train'):
          outputs = model(inputs)
          _, preds = torch.max(outputs, 1)
          loss = criterion(outputs, labels)
          if phase == 'train':
            loss.backward()
            optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels.data)

      epoch_loss = running_loss / dataset_sizes[phase]
      epoch_acc = running_corrects.double() / dataset_sizes[phase]

      print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')
  return model

criterion = nn.CrossEntropyLoss()

print("--- Training ResNet18 (Method 1: Frozen Weights) ---")
optimizer_res = optim.SGD(model_resnet.fc.parameters(), lr=0.001, momentum=0.9)
model_resnet = train_model(model_resnet, criterion, optimizer_res)

print("--- Training MobileNet V2 (Method 2: Full Fine-tuning) ---")
optimizer_mob = optim.SGD(model_mobilenet.parameters(), lr=0.001, momentum=0.9)
model_mobilenet = train_model(model_mobilenet, criterion, optimizer_mob)



--- Training ResNet18 (Method 1: Frozen Weights) ---
Epoch 0/2
train Loss: 0.4879 Acc: 0.7664
val Loss: 0.1845 Acc: 0.9412
Epoch 1/2
train Loss: 0.4293 Acc: 0.8074
val Loss: 0.2016 Acc: 0.9346
Epoch 2/2
train Loss: 0.4949 Acc: 0.7746
val Loss: 0.1596 Acc: 0.9477
--- Training MobileNet V2 (Method 2: Full Fine-tuning) ---
Epoch 0/2
train Loss: 0.5120 Acc: 0.7295
val Loss: 0.2068 Acc: 0.9281
Epoch 1/2
train Loss: 0.5031 Acc: 0.7746
val Loss: 0.4761 Acc: 0.7908
Epoch 2/2
train Loss: 0.7383 Acc: 0.6967
val Loss: 0.4790 Acc: 0.8170
